In [46]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]

err_p = df[:, 7]

err_q = df[:,8]

p = np.delete(p, [62, 74, 75, 76, 77, 88] )
q = np.delete(q, [62, 74, 75, 76, 77, 88] )
err_p = np.delete(err_p, [62, 74, 75, 76, 77, 88])
err_q = np.delete(err_q, [62, 74, 75, 76, 77, 88])




#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
Fit status: 0
Parametri stimati:
A = 3.7192471324942744e-13 ± 0.004329238562866822
B = 2.016521240879854 ± 0.049802794741526935
C = 200.7114593220005 ± 0.017625783135854493
Ch

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [47]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [48]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff

#residuals_norm = np.delete(residuals_norm, 0)

print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


[ 2.19204019e-01  6.77145141e-02  6.37710731e-02  5.00664047e-02
  5.47221787e-02  5.41927391e-02  5.44449527e-02  6.25234365e-02
  6.82908924e-02  5.47907239e-02  6.64275191e-02  8.05056949e-02
  5.77724261e-02  7.08358730e-02  6.18226544e-02  7.41462945e-02
  8.82817513e-02  7.47677183e-02  6.98527672e-02  8.65985026e-02
  8.35878766e-02  8.49623458e-02  6.26586150e-02  7.90415572e-02
  7.84756326e-02  7.17780500e-02  8.23462112e-02  6.73435541e-02
  8.65082269e-02  7.81745134e-02  8.00156379e-02  9.75730936e-02
  8.71903820e-02  6.73841393e-02  9.31348324e-02  9.35009926e-02
  7.72831417e-02  8.60474895e-02  7.28947787e-02  1.02328383e-01
  6.04172786e-02  6.31715220e-02  6.13546767e-02  7.83190577e-02
  5.63349035e-02  7.39800210e-02  7.22776560e-02  1.01772395e-01
  9.33782894e-02  8.69743816e-02  8.72455417e-02  8.69002455e-02
  1.43580783e-01  9.27522177e-02  1.13847055e-01  1.24936314e-01
  8.70069642e-02  1.06669019e-01  6.80960730e-02  7.71973649e-02
  1.17204361e-01 -3.57773

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [49]:
if np.abs(parameters[0] - 1 )< 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1] )< 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")
#print(3*errors[2])

if np.abs(parameters[2] - 1 )< 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [50]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 9.83842384e+01  9.11758929e+01  1.19625489e+02  8.75794855e+01
  8.81665429e+01  8.48188553e+01  8.33338914e+01  7.48322109e+01
  7.70196004e+01  6.62077356e+01  6.26865378e+01  7.22659314e+01
  5.51837357e+01  5.78793971e+01  5.05147719e+01  6.22909152e+01
  6.09585449e+01  5.76502680e+01  5.22527765e+01  5.18234271e+01
  4.61739401e+01  4.88887466e+01  4.54290590e+01  4.82107359e+01
  4.24468130e+01  3.96501923e+01  3.60113688e+01  3.33255170e+01
  3.68359221e+01  3.68859906e+01  3.31504772e+01  3.70558202e+01
  3.31127260e+01  3.17946428e+01  3.00111468e+01  2.90530955e+01
  2.75714039e+01  2.77273688e+01  2.60058189e+01  2.70854704e+01
  2.22496690e+01  2.39909638e+01  2.25948815e+01  1.71251155e+01
  1.49113796e+01  1.95818953e+01  1.66359078e+01  1.75684622e+01
  1.93433140e+01  1.70158145e+01  1.70688647e+01  1.90014639e+01
  1.48713875e+01  1.70787769e+01  1.57222825e+01  1.72537095e+01
  1.60208841e+01  1.59585775e+01  1.25387583e+01  1.24377842e+01
  1.21394482e+01 -1.64746

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
